# LaGoVAD-OSTP Step 1 — patch-only local branch **L1** vs a free ensemble **L0b**, train Nexar → DoTA-CAP-dev, 1 seed

Rule (written before any number): `core/docs/final/LOCAL_STEP1.md`. Question: does a patch-only branch trained **alone**
add to the host what a **second host seed** does not?

| row | what | seed | here |
|---|---|---|---|
| **L0** | host = W0 (`WU_ARMS.md`), SA² off | 2024 | **reused**: `Thesis-Final/outputs/wu/W0/s2024` + its CAP-dev read |
| **L0b** | W0's exact command | 2025 | trained |
| **L1** | `stp.sa2=true`, `stp.branch=local`: top-12 patches → SA² (CLS = query only, no `u`) → own RoFormer → one conv head, **no text** | 2024 | trained |

Fusion fixed: `0.5·minmax(a) + 0.5·minmax(b)` per clip. Gates: **G1** L1 alone low > 0.60 and mean > 0.7132;
**G2** (L0+L1) − (L0+L0b) gain shown; **G3** oracle max(L0,L1) − max(L0,L0b) low > 0. Every sentence is
**PRELIMINARY (n = 1 seed)**.

| step | what | GPU? |
|---|---|---|
| 1–3 | setup, deps, helpers | — |
| 4 | **train** L0b, L1 (1,749 steps each) + config provenance vs W0 | **yes** |
| 5 | protocol B on DoTA-CAP-dev: L0b, L1 (L0 reused; re-scored only if its `clip_scores.npz` is missing) | yes |
| 6 | **read-out** → paste `REPORTS/local_readout.md` to Claude | no |

**Inputs reused, nothing re-extracted:** the STP patch caches (`Thesis-Final/cache/clip_patches/…`), v2's A2 bakes,
Nexar_window, KNN. **Drive:** code `Thesis-Final/kat-vad` (upload branch **`v_final-local`**); outputs under
`Thesis-Final/outputs/local/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, build report, p_T2
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + STP patch caches + W0 run (read) + everything written here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final-local there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'LOCAL_ARMS'), 'this upload predates Step 1 -- upload branch v_final-local'
assert (REPO / 'core/docs/final/LOCAL_STEP1.md').is_file(), 'no LOCAL_STEP1.md (the read-out rule) in this upload'

L0, L0B, L1 = constants.LOCAL_CONTROL_ARM, constants.LOCAL_ENSEMBLE_ARM, constants.LOCAL_ARM
TRAIN    = [L0B, L1]                  # L0 is W0 s2024, reused
SEED     = constants.LOCAL_SEEDS      # {'L0': 2024, 'L0b': 2025, 'L1': 2024}
ENC      = constants.V5_ENCODER       # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_N, STRIDE_D, K, MOTION = (constants.STP_NEXAR_STRIDE, constants.STP_DOTA_STRIDE,
                                 constants.STP_TOP_K, constants.STP_MOTION_RAW)

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                         # host rows [x ; c u~/sigma_u]
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
PRIOR    = V2_OUT / 'v2_e3' / 'REPORTS' / 'position_prior' / CAP_DEV / 'position_prior.npz'   # p_T2 (printed only)

from core.data import stp_patches as stp_data      # noqa: E402

PATCH_N  = stp_data.cache_dir(DATASET, STRIDE_N, MOTION, K)                  # built by stp_nexar_dotacap.ipynb
PATCH_D  = stp_data.cache_dir(constants.DOTA_CAP_DATASET, STRIDE_D, MOTION, K)
WU_ROOT  = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME
W0_RUN   = WU_ROOT / constants.WU_CONTROL_ARM / f's{SEED[L0]}'               # L0 = W0 s2024
W0_PB    = WU_ROOT / 'eval' / 'dota' / CAP_DEV / constants.WU_CONTROL_ARM
ROOT     = constants.OUTPUT_ROOT / constants.LOCAL_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/local')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('A2 bake', A2_BAKE), ('KNN', KNN), ('DoTA s1', DOTA_S1),
                   ('DoTA-CAP-dev A2 bake', DOTA_A2), ('DoTA labels', DOTA / 'labels_s8'),
                   ('Nexar patches', PATCH_N), ('DoTA-CAP-dev patches', PATCH_D),
                   ('W0 s2024 run', W0_RUN / 'checkpoint_last.pt'), ('W0 config', W0_RUN / 'config.yaml')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}')
print(f'  p_T2 prior            {PRIOR if PRIOR.exists() else "absent -- position rho not printed"}')
print(f'TRAIN {TRAIN} | seeds {SEED} | steps {STEPS} ({EPOCHS} epochs)')
print('runs/reads ->', ROOT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 4-7 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        tmp = dst.with_name(dst.name + '.partial')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        os.replace(tmp, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.partial')
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi))."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


from core.data.v2_splits import load_split    # noqa: E402

CAP_IDS = sorted(load_split(CAP_DEV))
assert len(CAP_IDS) == 569, len(CAP_IDS)
print(f'{len(CAP_IDS)} DoTA-CAP-dev clips')

from core.tools.stp_patches import training_sources   # noqa: E402

SOURCES = sorted(training_sources(DATA))
missing_n = [v for v in SOURCES if not (PATCH_N / f'{v}.npy').exists()]
missing_d = [v for v in CAP_IDS if not (PATCH_D / f'{v}.npy').exists()]
assert not missing_n, f'{len(missing_n)} Nexar sources have no patch file, e.g. {missing_n[:3]} -- run stp_nexar_dotacap.ipynb step 4'
assert not missing_d, f'{len(missing_d)} CAP-dev clips have no patch file, e.g. {missing_d[:3]} -- run stp_nexar_dotacap.ipynb step 5'
print(f'patch caches complete: {len(SOURCES)} Nexar sources, {len(CAP_IDS)} CAP-dev clips')

## 4. Train L0b and L1 (GPU), then config provenance vs W0

Exactly W0's command (v2 Nexar A2's recipe) on `train.seed` from `LOCAL_SEEDS`, plus L1's `stp.sa2=true`,
`stp.branch=local` and `--patch-dir`. Local training, atomic publish to Drive, J10 re-verified after a flush, then
`verified.json`; a verified run is never retrained. **Provenance:** each config must equal W0 s2024's except
`train.seed` (L0b) or `stp.sa2` / `stp.branch` (L1).

In [ ]:
INPUTS = {'data': (DATA, LOCAL / 'stage' / f'data_{C}'), 'a2': (A2_BAKE, LOCAL / 'stage' / 'A2_s8'),
          'knn': (KNN, LOCAL / 'stage' / f'knn_{C}.npz'), 'patch_n': (PATCH_N, LOCAL / 'stage' / 'patch_nexar')}
_STAGED = {}


def staged(key):
    if key not in _STAGED:
        _STAGED[key] = stage(*INPUTS[key])
    return _STAGED[key]


def arm_sets(arm):
    return [x for k, v in constants.LOCAL_ARMS[arm].items() for x in ('--set', f'{k}={str(v).lower()}')]


def train_cmd(arm, seed, out_dir):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={seed}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}', *arm_sets(arm),
           '--data-dir', staged('data'), '--clip-dir', staged('a2'), '--knn-cache', staged('knn'),
           '--output-dir', out_dir]
    if constants.LOCAL_ARMS[arm].get('stp.sa2'):
        cmd += ['--patch-dir', staged('patch_n')]
    return cmd


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def train_arm(arm, seed):
    drive_dir, local = ROOT / arm / f's{seed}', LOCAL / 'runs' / arm / f's{seed}'
    if is_verified(drive_dir):
        print(f'{arm} s{seed}: verified on Drive -- not retrained')
        return
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(arm, seed, local) + resume, log=LOCAL / f'train_{arm}_s{seed}.log', quiet_steps=True)
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm} s{seed}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm} s{seed}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm} s{seed}: {(time.time() - t0) / 60:.0f} min | last mil {last.get("mil"):.4f} '
          f'(W0 {constants.LOCAL_W0_MIL_LAST}) | SA² entropy {last.get("stp_entropy")} | rho_u {last.get("motion_share")}')


assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
assert ckpt_step(W0_RUN / 'checkpoint_last.pt') == STEPS, f'L0 = W0 s{SEED[L0]} is not a finished {STEPS}-step run'
for arm in TRAIN:
    train_arm(arm, SEED[arm])

from core.tools.v5_stage1 import _flat   # noqa: E402

ref = _flat(yaml.safe_load((W0_RUN / 'config.yaml').read_text()))
NEW_DEFAULTS = {'stp.branch': constants.STP_BRANCH_HOST}   # keys this branch added; absent in W0's config
for arm in TRAIN:
    got = _flat(yaml.safe_load((ROOT / arm / f's{SEED[arm]}' / 'config.yaml').read_text()))
    allowed = {'train.seed', *constants.LOCAL_ARMS[arm]}
    diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got) if k not in allowed and ref.get(k) != got.get(k)
            and not (k not in ref and got.get(k) == NEW_DEFAULTS.get(k))}   # W0 predates stp.branch
    assert not diff, f'{arm} differs from W0 s{SEED[L0]} beyond {sorted(allowed)}: {diff}'
    assert got['train.seed'] == SEED[arm], got['train.seed']
    assert all(got[k] == v for k, v in constants.LOCAL_ARMS[arm].items()), (arm, {k: got[k] for k in constants.LOCAL_ARMS[arm]})
    print(f'{arm}: config == W0 s{SEED[L0]} except', {k: got[k] for k in sorted(allowed)})

## 5. DoTA-CAP-dev — protocol B (GPU)

`s1[::3]`, whole clip, native frames, macro over two-class clips; input = v2's A2 bake of CAP-dev (+ L1's patch
cache). L0 reuses W0's read (`Thesis-Final/outputs/wu/eval/…/W0`); if its `clip_scores.npz` is missing, W0 s2024 is
re-scored here into `eval/…/L0`. Skipped when a read already exists for the row's seed.

In [ ]:
PB_OUT = ROOT / 'eval' / 'dota' / CAP_DEV
D_INPUTS = {'dota_a2': (DOTA_A2, LOCAL / 'stage' / 'dota_a2', CAP_IDS),
            'dota_s1': (DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
            'dota_lab': (DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels', None),
            'patch_d': (PATCH_D, LOCAL / 'stage' / 'patch_dota', CAP_IDS)}


def d_staged(key):
    if key not in _STAGED:
        _STAGED[key] = stage(*D_INPUTS[key])
    return _STAGED[key]


def read_done(out, seed):
    path = out / 'protocol_b_readout.json'
    if not (path.exists() and (out / 'clip_scores.npz').exists()):
        return False
    return sorted(json.loads(path.read_text()).get('per_run_macro', {})) == [f's{seed}']


def score(arm, ckpt, seed, out, patches):
    local, t0 = LOCAL / 'dota' / arm, time.time()
    extra = ['--patch-dir', d_staged('patch_d')] if patches else []
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{seed}', ckpt,
         '--input-dir', d_staged('dota_a2'), '--s1-dir', d_staged('dota_s1'), *extra,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', d_staged('dota_lab'), '--split', CAP_DEV, '--out-dir', local, '--device', 'auto'],
        log=LOCAL / f'dota_{arm}.log')
    publish(local, out)
    r = json.loads((local / 'protocol_b_readout.json').read_text())
    print(f"dota {arm}: macro {r['macro']['mean']:.4f} ({(time.time() - t0) / 60:.1f} min)")


wrote = False
L0_PB = W0_PB if read_done(W0_PB, SEED[L0]) else PB_OUT / L0
if not read_done(L0_PB, SEED[L0]):
    print('W0 read has no clip_scores.npz -- re-scoring W0 s2024 as L0')
    score(L0, W0_RUN / 'checkpoint_last.pt', SEED[L0], L0_PB, patches=False)
    wrote = True
for arm in TRAIN:
    out = PB_OUT / arm
    if read_done(out, SEED[arm]):
        print(f'dota {arm}: already read')
        continue
    score(arm, ROOT / arm / f's{SEED[arm]}' / 'checkpoint_last.pt', SEED[arm], out,
          patches=bool(constants.LOCAL_ARMS[arm].get('stp.sa2')))
    wrote = True
if wrote:
    flush_drive()
for arm, out in ((L0, L0_PB), (L0B, PB_OUT / L0B), (L1, PB_OUT / L1)):
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    print(f"{arm:4s} macro {r['macro']['mean']:.4f} [{r['macro']['low']:.4f}, {r['macro']['high']:.4f}]  arm {r['arm']}  <- {out}")

## 6. Read-out → **paste `REPORTS/local_readout.md` to Claude**

Rule: `LOCAL_STEP1.md` §4–§5. CPU only. Verdict from the decision table; every sentence **PRELIMINARY (n = 1 seed)**.

In [ ]:
prior = ['--position', PRIOR] if PRIOR.exists() else []
run([sys.executable, '-m', 'core.tools.local_readout',
     '--l0', L0_PB, '--l0b', PB_OUT / L0B, '--l1', PB_OUT / L1,
     '--l0-run', W0_RUN, '--l0b-run', ROOT / L0B / f's{SEED[L0B]}', '--l1-run', ROOT / L1 / f's{SEED[L1]}',
     *prior, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
     '--out-dir', REPORTS])
print((REPORTS / 'local_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/local_readout.md to Claude')